In [ ]:

from markovstates.utils import X
import numpy as np
import pandas as pd

n_samples, n_features = X.shape
ratio = n_features / n_samples

"""
btw quick note for Raul here: This part was not claude coded, as nuanced the math may seem.

I remember reading about RMT for Covariance matrix decomposition a while back in a quantitative finance newsletter I follow,
and I figured it would be quite quite useful here (it was!)
"""

# Rough autocorrelation correction
autocorr = pd.DataFrame(X).apply(lambda col: col.autocorr(lag=1)).mean()
n_eff = n_samples * (1 - autocorr) / (1 + autocorr)

lambda_max_corrected = (1 + np.sqrt(n_features / n_eff)) ** 2

# Get eigenvalues of the correlation matrix
eigenvalues = np.linalg.eigvalsh(np.corrcoef(X.T))
eigenvalues = np.sort(eigenvalues)[::-1]

# Count how many are above the noise threshold
n_signal = np.sum(eigenvalues > lambda_max_corrected)
print(f"Signal components: {n_signal}")
print(f"Noise threshold: {lambda_max_corrected:.3f}")
print(eigenvalues)

FINAL_FEATURES = [
    "apparent_temperature",   # Factor 1 anchor
    "wind_speed_10m",         # Factor 2 anchor
    "dew_point_2m",           # Factor 3 anchor
]

# ok from our RMT assumptions, we see that just that alone suggests to only use 
# 2 components for FA, but interpreting the physical data, we find that underlying 
# factor 3 is probably better, and produces more meaningful results. either way, RMT is
# not deterministic, and the 3rd eigenvalue isn't terribly below the noise threshold, so it doesn't hurt
# to have 3 features

Signal components: 2
Noise threshold: 1.443
[2.90977347 1.79115333 1.13887496 0.84977366 0.52309306 0.43002457
 0.2598593  0.09744765]


In [3]:
# looking at the factor analysis results, it seems that the temperature_2m and apparent_temperature variables 
# were correlating to each other and giving them an overestimated factor strength;  we shall remove one of them and 
# run the code again
import numpy as np
import pandas as pd
from markovstates.utils import hourly_dataframe

onetempdf = hourly_dataframe.drop('date', axis=1)
feat_cols = onetempdf.columns

from sklearn.preprocessing import StandardScaler
sclr = StandardScaler()
sclr.fit(onetempdf[feat_cols])
normalized_df = sclr.transform(onetempdf[feat_cols]).astype(np.float64)

print(normalized_df)

# factor analysis done again:

from sklearn.decomposition import FactorAnalysis
fa = FactorAnalysis(n_components=3)
fa.fit(normalized_df)

loadings = pd.DataFrame(
    fa.components_.T,
    index=feat_cols,
    columns=["Factor 1", "Factor 2", "Factor 3"]
)
print(loadings.round(2))

# yep, much better readings now

[[ 0.15332715 -1.63558328 -0.17689671 ... -0.20414129  1.14834869
  -0.62193793]
 [-0.0388618  -1.55871975 -0.17689671 ... -0.47387272  1.03468323
  -0.62193793]
 [-0.17064849 -1.48185635 -0.17689671 ... -0.44911298  1.14486289
  -0.62193793]
 ...
 [ 0.63654506 -1.25126612 -0.17689671 ... -0.61934716 -0.03125098
  -0.62193793]
 [ 0.47181165 -1.20734417 -0.17689671 ... -0.8180905  -0.07257479
  -0.62193793]
 [ 0.29609621 -1.05361724 -0.17689671 ... -1.24547076 -0.24764733
  -0.62193793]]
                    Factor 1  Factor 2  Factor 3
temperature_2m          0.97      0.07      0.04
dew_point_2m            0.42     -0.77     -0.26
precipitation          -0.07     -0.27      0.20
surface_pressure        0.06      0.42     -0.71
cloud_cover_mid        -0.19     -0.35      0.37
wind_speed_10m          0.07     -0.14      0.50
wind_direction_10m      0.06     -0.06      0.27
direct_radiation        0.51      0.33      0.04


In [4]:
from sklearn.decomposition import FactorAnalysis
import pandas as pd
import numpy as np
from markovstates.preprocessing import fit_scaler, apply_scaler, extract_features
from markovstates.data_collect import hourly_dataframe

# constructing variables

feature_cols = extract_features(hourly_dataframe)
scaler = fit_scaler(hourly_dataframe, feature_cols)
feature_mat = apply_scaler(hourly_dataframe, feature_cols, scaler)
X = feature_mat

fa = FactorAnalysis(n_components=3)
fa.fit(X)

loadings = pd.DataFrame(
    fa.components_.T,
    index=feature_cols,
    columns=["Factor 1", "Factor 2", "Factor 3"]
)
print(loadings.round(2))

# after factor analysis, we find that the vars contributing the most real variance are the ones below:

FINAL_FEATURES = [
    "apparent_temperature",   # Factor 1 anchor
    "wind_speed_10m",         # Factor 2 anchor
    "dew_point_2m",           # Factor 3 anchor
]

# But wait:

'''
After further analysis, we find in the factor_explore notebook that 
the variables apparent_temperature, and temperature_2m were correlating to each
other since they were more or less the same thing, strengthening each other's signal 
too much (and thus messing with other variables' influence on the factors as well)

We shall remove apparent_temperature from the df, and review the FA results again
'''


ImportError: cannot import name 'fit_scaler' from 'markovstates.preprocessing' (/Users/philipalexopoulos/markovstates/markovstates/preprocessing.py)